In [1]:
import pandas as pd

In [2]:
application = pd.read_csv("application_record.csv")
credit = pd.read_csv("credit_record.csv")

print(application.shape)
print(credit.shape)

(438557, 18)
(1048575, 3)


In [3]:
common_ids = set(application['ID']).intersection(set(credit['ID']))

len(common_ids)

36457

In [4]:
application_filtered = application[application['ID'].isin(common_ids)]
credit_filtered = credit[credit['ID'].isin(common_ids)]

In [5]:
print(application_filtered.shape)
print(credit_filtered.shape)

(36457, 18)
(777715, 3)


In [6]:
credit_filtered['STATUS'].value_counts()

,count
STATUS,
C,329536
0,290654
X,145950
1,8747
5,1527
2,801
3,286
4,214


In [7]:
credit_filtered['STATUS'] = credit_filtered['STATUS'].replace({
    'C': 0,
    'X': 0
})

/tmp/ipykernel_227/598956084.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  credit_filtered['STATUS'] = credit_filtered['STATUS'].replace({


In [8]:
credit_filtered['STATUS'] = credit_filtered['STATUS'].astype(int)

/tmp/ipykernel_227/1193234651.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  credit_filtered['STATUS'] = credit_filtered['STATUS'].astype(int)


In [9]:
credit_summary = credit_filtered.groupby('ID')['STATUS'].max().reset_index()

In [10]:
credit_summary.rename(columns={'STATUS': 'MAX_DELAY'}, inplace=True)

In [11]:
credit_summary.head()

,ID,MAX_DELAY
0,5008804,1
1,5008805,1
2,5008806,0
3,5008808,0
4,5008809,0


In [12]:
credit_summary['TARGET'] = credit_summary['MAX_DELAY'].apply(lambda x: 1 if x >= 2 else 0)

In [13]:
credit_summary['TARGET'].value_counts()

,count
TARGET,
0,35841
1,616


In [14]:
final_dataset = application_filtered.merge(credit_summary, on='ID', how='inner')

In [15]:
final_dataset.shape

(36457, 20)

In [16]:
final_dataset.head()

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,MAX_DELAY,TARGET
0,5008804,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,0
1,5008805,M,Y,Y,0,427500.0,Working,Higher education,Civil marriage,Rented apartment,-12005,-4542,1,1,0,0,NaN,2.0,1,0
2,5008806,M,Y,Y,0,112500.0,Working,Secondary / secondary special,Married,House / apartment,-21474,-1134,1,0,0,0,Security staff,2.0,0,0
3,5008808,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0,0
4,5008809,F,N,Y,0,270000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-19110,-3051,1,0,1,1,Sales staff,1.0,0,0


In [17]:
final_dataset.to_csv("final_dataset.csv", index=False)